# Zero-Shot Foundation Models for Brazilian Electricity Load Forecasting

This notebook demonstrates the core experiment: **Chronos-2** (Amazon, 120M params) predicting day-ahead electricity demand for Brazil's SE (Sudeste) subsystem — zero-shot, no training on Brazilian data.

**Key result:** 1.86% MAPE, matching US ISO-grade accuracy (PJM: 1.78-1.98%).

For the full benchmark (all models, all subsystems, horizon sensitivity, probabilistic evaluation), see `scripts/benchmark.py`.

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, root_mean_squared_error,
    mean_absolute_percentage_error, r2_score,
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Auto-detect best available device
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Device: {DEVICE}')

## 2. Load ONS Data

We use hourly electricity load data from ONS (Operador Nacional do Sistema Eletrico),
Brazil's grid operator. The data is freely available under CC-BY 4.0.

**If you haven't downloaded the data yet**, run:
```bash
python scripts/download_ons.py --subsystem SE
```

In [ ]:
# Load the processed SE subsystem data
import glob
files = glob.glob(os.path.join('..', 'data', 'processed', 'ons_hourly_load_se_*.csv'))
df = pd.read_csv(files[0])
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

print(f'Rows:       {len(df):,}')
print(f'Date range: {df["datetime"].min()} to {df["datetime"].max()}')
print(f'Mean load:  {df["load_mw"].mean():,.0f} MW')
print(f'Min load:   {df["load_mw"].min():,.0f} MW')
print(f'Max load:   {df["load_mw"].max():,.0f} MW')

### 2.1 Visualize the load profile

Electricity demand follows strong daily (24h) and weekly (168h) cycles.
These patterns are driven by human behaviour and physics — not market dynamics —
which is why they're predictable.

In [ ]:
# Plot last 4 weeks to see daily and weekly patterns
last_4w = df.tail(24 * 28)

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(last_4w['datetime'], last_4w['load_mw'], 'b-', linewidth=0.8)
axes[0].set_ylabel('Load (MW)')
axes[0].set_title('SE Subsystem — Last 4 Weeks (daily cycles visible)')
axes[0].grid(True, alpha=0.3)

# Average load by hour of day
df['hour'] = df['datetime'].dt.hour
hourly_avg = df.groupby('hour')['load_mw'].mean()
axes[1].bar(hourly_avg.index, hourly_avg.values, color='steelblue')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Avg Load (MW)')
axes[1].set_title('Average Load by Hour — Clear daily cycle (peak ~15:00, trough ~04:00)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 3. Train/Test Split

We use the most recent 365 days as the test set. Everything before that
serves as context for the foundation model (which sees it as input, not training data).

In [ ]:
TEST_DAYS = 365
CONTEXT_LENGTH = 720  # 30 days of hourly data
HORIZON = 24          # predict next 24 hours (day-ahead)

test_cutoff = df['datetime'].max() - pd.Timedelta(days=TEST_DAYS)
train_df = df[df['datetime'] <= test_cutoff]
test_df = df[df['datetime'] > test_cutoff]

full_load = df['load_mw'].values.astype(np.float32)
test_start_idx = len(train_df)
test_actuals = full_load[test_start_idx:]
test_dates = df['datetime'].values[test_start_idx:]

print(f'Train: {len(train_df):,} hours ({len(train_df)//24:,} days)')
print(f'Test:  {len(test_df):,} hours ({TEST_DAYS} days)')
print(f'Context window: {CONTEXT_LENGTH} hours ({CONTEXT_LENGTH//24} days)')
print(f'Forecast horizon: {HORIZON} hours')

## 4. Naive Baseline

Before running the foundation model, we establish the baseline to beat:
**repeat the same hour from exactly 7 days ago**.

This captures weekly seasonality and is a standard baseline in load forecasting.
It's not trivial — weekly patterns are strong in electricity demand.

In [ ]:
# Naive: same hour, 7 days ago (168 hours)
naive_preds = []
for i in range(0, len(test_actuals), HORIZON):
    actual_h = min(HORIZON, len(test_actuals) - i)
    for h in range(actual_h):
        idx = test_start_idx + i + h - 168  # 7 * 24
        naive_preds.append(full_load[idx])
naive_preds = np.array(naive_preds[:len(test_actuals)])

print(f'Naive predictions: {len(naive_preds):,} hours')

## 5. Chronos-2 Zero-Shot Forecast

**Chronos-2** (Amazon, 120M parameters) is an encoder-only transformer pre-trained on
100B+ time points from diverse domains (weather, retail, energy, finance, sensors).

It has **never seen Brazilian electricity data**. We give it 30 days of context
and ask it to predict the next 24 hours. This is repeated across the entire test year
in a rolling fashion.

In [ ]:
from chronos import BaseChronosPipeline

model = BaseChronosPipeline.from_pretrained(
    'amazon/chronos-2',
    device_map=DEVICE,
    dtype=torch.float32,
)
print('Chronos-2 loaded.')

In [ ]:
# Rolling day-ahead forecast across the test year
chronos_preds = []
for i in tqdm(range(0, len(test_actuals), HORIZON), desc='Chronos-2'):
    # Context: last 30 days of actual load
    ctx = torch.tensor(
        full_load[test_start_idx + i - CONTEXT_LENGTH : test_start_idx + i],
        dtype=torch.float32,
    ).reshape(1, 1, -1)  # Chronos-2 expects (n_series, n_variates, length)

    actual_h = min(HORIZON, len(test_actuals) - i)
    forecasts = model.predict(ctx, prediction_length=actual_h)

    # Extract median forecast (middle quantile)
    f = forecasts[0]  # shape: (n_variates, n_quantiles, pred_len)
    median_idx = f.shape[1] // 2
    chronos_preds.extend(f[0, median_idx, :actual_h].tolist())

chronos_preds = np.array(chronos_preds[:len(test_actuals)])
print(f'Predictions: {len(chronos_preds):,} hours')

## 6. Evaluate

We compute the same metrics used in load forecasting literature:
- **MAPE**: Mean Absolute Percentage Error (the headline number)
- **MAE**: Mean Absolute Error in MW
- **RMSE**: Root Mean Squared Error in MW
- **MASE**: Mean Absolute Scaled Error (< 1.0 means better than naive)
- **R²**: Coefficient of determination

In [ ]:
def compute_metrics(actual, predicted, name):
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    naive_err = mean_absolute_error(actual[24:], actual[:-24])  # seasonal naive denominator
    metrics = {
        'MAE (MW)':  mean_absolute_error(actual, predicted),
        'RMSE (MW)': root_mean_squared_error(actual, predicted),
        'MAPE':      mean_absolute_percentage_error(actual, predicted) * 100,
        'MASE':      mean_absolute_error(actual, predicted) / naive_err,
        'R²':        r2_score(actual, predicted),
    }
    return metrics

naive_metrics = compute_metrics(test_actuals, naive_preds, 'Naive')
chronos_metrics = compute_metrics(test_actuals, chronos_preds, 'Chronos-2')

results = pd.DataFrame({'Naive (7d ago)': naive_metrics, 'Chronos-2 (zero-shot)': chronos_metrics}).T
results = results.round(2)

# Format MAPE as percentage
display = results.copy()
display['MAPE'] = display['MAPE'].apply(lambda x: f'{x:.2f}%')
display

## 7. Visualize Predictions

First 7 days of the test set: actual load vs Chronos-2 vs naive baseline.

In [ ]:
n_plot = 168  # 7 days

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

# Time series comparison
axes[0].plot(test_dates[:n_plot], test_actuals[:n_plot], 'k-', lw=1.5, label='Actual')
axes[0].plot(test_dates[:n_plot], chronos_preds[:n_plot], 'b--', lw=1, alpha=0.8, label='Chronos-2')
axes[0].plot(test_dates[:n_plot], naive_preds[:n_plot], 'r--', lw=1, alpha=0.5, label='Naive (7d ago)')
axes[0].set_ylabel('Load (MW)')
axes[0].set_title('SE Subsystem — First 7 Days of Test Set')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# Error distribution
chronos_errors = (chronos_preds - test_actuals) / test_actuals * 100
naive_errors = (naive_preds - test_actuals) / test_actuals * 100
axes[1].hist(chronos_errors, bins=80, alpha=0.7, label=f'Chronos-2 (std={chronos_errors.std():.1f}%)', color='blue')
axes[1].hist(naive_errors, bins=80, alpha=0.4, label=f'Naive (std={naive_errors.std():.1f}%)', color='red')
axes[1].set_xlabel('Percentage Error (%)')
axes[1].set_ylabel('Count')
axes[1].set_title('Error Distribution — Chronos-2 is tighter and centered near zero')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Context: How Good Is This?

Comparing our zero-shot result against published benchmarks worldwide.

In [ ]:
comparison = pd.DataFrame([
    {'Benchmark': 'PJM (US) official',      'MAPE': 1.88, 'Method': 'Proprietary'},
    {'Benchmark': 'Our Chronos-2 on SE',     'MAPE': chronos_metrics['MAPE'], 'Method': 'Zero-shot'},
    {'Benchmark': 'N-BEATS on Portugal',     'MAPE': 1.90, 'Method': 'Trained DL'},
    {'Benchmark': 'ERCOT (US) official',     'MAPE': 2.70, 'Method': 'Proprietary'},
    {'Benchmark': 'ANN on Europe (4 countries)', 'MAPE': 2.80, 'Method': 'Trained NN'},
    {'Benchmark': 'Our Naive baseline',      'MAPE': naive_metrics['MAPE'], 'Method': 'Baseline'},
]).set_index('Benchmark').sort_values('MAPE')

colors = ['#FF5722' if 'Our' in idx else '#2196F3' if 'Naive' in idx else '#9E9E9E'
          for idx in comparison.index]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(comparison.index, comparison['MAPE'], color=colors)
ax.set_xlabel('MAPE (%)')
ax.set_title('Day-Ahead Load Forecasting — International Comparison')
ax.grid(True, alpha=0.3, axis='x')

for i, (idx, row) in enumerate(comparison.iterrows()):
    ax.text(row['MAPE'] + 0.05, i, f"{row['MAPE']:.2f}% ({row['Method']})", va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Summary

| Finding | Detail |
|---------|--------|
| **Chronos-2 MAPE** | ~1.86% on SE (zero-shot, no Brazilian training data) |
| **vs Naive** | 64% better (5.13% → 1.86%) |
| **vs PJM (US)** | Comparable (PJM: 1.78-1.98% with proprietary system) |
| **vs trained Linear** | 18% better (Linear trained on 5+ years: 2.26%) |
| **Multi-year stability** | 1.89% ± 0.04% across 2023/2024/2025 |
| **All 4 subsystems** | 1.67-3.17% MAPE, R² > 0.90 everywhere |

**Bottom line:** A foundation model that never saw Brazilian data matches the accuracy of
purpose-built forecasting systems used by the largest US grid operators.

For the full analysis (all models, horizons, probabilistic evaluation), see:
- `scripts/benchmark.py` — all models, all subsystems
- `scripts/probabilistic_eval.py` — CRPS, calibration, prediction intervals
- `scripts/train_baselines.py` — trained model comparison
- `DRAFT_PAPER.md` — full paper draft